# DecisionAI: обучение модели D на GPU

Рецепт как у модели C (INIT=hf, JEVA=0.5, ALPHA=0.15, MAXTEXTS=80, 3 эпохи),
но на чистом корпусе v2 + отфильтрованные мягкие учителя (v2extra). На T4 это минуты вместо ~10 часов на CPU.
Выполняй ячейки по порядку.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%pip install -q transformers safetensors
import torch
print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())

In [ ]:
from google.colab import files
import shutil, pathlib
up = files.upload()  # выбери train-D-bundle.zip
name = next(iter(up))
shutil.unpack_archive(name, '/content/traind')
print(sorted(str(p) for p in pathlib.Path('/content/traind').rglob('*') if p.is_file())[:8])

In [ ]:
import pathlib, urllib.request
dst = pathlib.Path('/content/traind/data/hf/turbo')
dst.mkdir(parents=True, exist_ok=True)
base = 'https://huggingface.co/sergeyzh/rubert-tiny-turbo/resolve/main/'
for f in ('config.json', 'tokenizer.json', 'tokenizer_config.json',
          'special_tokens_map.json', 'vocab.txt', 'model.safetensors'):
    o = dst / f
    if not o.exists():
        print('get', f, flush=True)
        urllib.request.urlretrieve(base + f, o)
    print('ok', f, o.stat().st_size)

In [ ]:
import os, torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev, '-', torch.cuda.get_device_name(0) if dev == 'cuda' else 'без GPU: будет медленно, лучше включи GPU в Среда выполнения')
os.environ.update({
    'INIT': 'hf', 'JEVA': '0.5', 'ALPHA': '0.15', 'MAXTEXTS': '80',
    'EXTRA': '/content/traind/data/prepared/v2extra/syn2.para.plans.jsonl,/content/traind/data/prepared/v2extra/syn3.para.plans.jsonl,/content/traind/data/prepared/v2extra/syn4.para.plans.jsonl,/content/traind/data/prepared/v2extra/train.para.plans.jsonl,/content/traind/data/prepared/v2extra/syn5.gemma.plans.jsonl,/content/traind/data/prepared/v2extra/syn6.qwen.plans.jsonl,/content/traind/data/prepared/v2extra/syn6.gemma.plans.jsonl',
    'PYTHONUTF8': '1', 'TORCH_DEVICE': dev,
})
%cd /content/traind/scripts/distill
!python -u /content/traind/scripts/distill/train_embed.py /content/traind/data/hf/turbo /content/traind/models /content/traind/data/prepared/v2 /content/traind/models/s1-distilled-d2.dte 3

In [ ]:
from google.colab import files
files.download('/content/traind/models/s1-distilled-d2.dte')

# Часть 2: свежие мягкие метки Gemma-2 под v2 и модель D3

Учитель заново размечает v2-корпус (старые syn5/syn6.gemma — v1-эпоха, частично вычищены фильтром).
Запускать после части 1, в том же рантайме.


In [ ]:
import pathlib, urllib.request
dst = pathlib.Path('/content/traind/gemma2')
dst.mkdir(parents=True, exist_ok=True)
base = 'https://huggingface.co/google/embeddinggemma-2/resolve/main/'
for f in ('config.json', 'tokenizer.json', 'tokenizer.model',
          'tokenizer_config.json', 'model.safetensors'):
    o = dst / f
    print(('have ' if o.exists() else 'get ') + f, flush=True)
    if not o.exists():
        urllib.request.urlretrieve(base + f, o)
    print('ok', f, o.stat().st_size)


In [ ]:
%pip install -q -U transformers
import transformers
print('transformers', transformers.__version__)


In [ ]:
%cd /content/traind/scripts/distill
!python -u /content/traind/scripts/distill/gemma_cache.py /content/traind/gemma2 /content/traind/gemma-cache-v2 /content/traind/data/prepared/v2/train.plans.jsonl /content/traind/data/prepared/v2/dev.plans.jsonl /content/traind/data/prepared/v2extra/*.plans.jsonl


In [ ]:
!JEVA=0.5 ALPHA=0.15 python -u train_gemma_head.py /content/traind/gemma-cache-v2 /content/traind/data/prepared/v2 /content/traind/models/gemma-head-v2.pt 8


In [ ]:
from google.colab import files
up = files.upload()  # залей models/gemma-head.pt из проекта (3.6 МБ)
import shutil
shutil.move(next(iter(up)), '/content/traind/models/gemma-head.pt')
print('head ok')


In [ ]:
import os
os.environ.update({'PREDICT': '/content/traind/data/prepared/v2/train.plans.jsonl',
                   'OUT_LABELS': '/content/traind/data/prepared/v2/v2.gemma.plans.jsonl'})
!python -u /content/traind/scripts/distill/train_gemma_head.py /content/traind/gemma-cache-v2 /content/traind/data/prepared/v2 /content/traind/models/gemma-head.pt


In [ ]:
# склейка: train.plans + мягкие метки головы -> soft-планы
import hashlib, json as js, pathlib, sys
sys.path.insert(0, '/content/traind/scripts/distill')
from data_checks import signatures
V2 = pathlib.Path('/content/traind/data/prepared/v2')
lab = {}
for l in open(V2 / 'v2.gemma.plans.jsonl', encoding='utf-8'):  # OUT_LABELS: карта group->метки
    l = l.strip()
    if l:
        d = js.loads(l)
        lab[d['group']] = d['labels']
print('label groups:', len(lab))
n0 = n1 = 0
out = []
for l in open(V2 / 'train.plans.jsonl', encoding='utf-8'):
    l = l.strip()
    if not l:
        continue
    r = js.loads(l)
    m = lab.get(r.get('group'))
    if not m:
        continue
    qs = []
    for q in r['questions']:
        if q['id'] in m:
            q = dict(q)
            q['label'] = m[q['id']]
            qs.append(q)
    if qs:
        r = dict(r)
        r['questions'] = qs
        out.append(r)
        n1 += len(qs)
    n0 += 1
print(f'rows: {len(out)}/{n0}, soft questions: {n1}')
# чистка от протечек в dev/test (группы + чанки)
dg, ds, _ = signatures([V2 / 'dev.plans.jsonl'])
tg, ts, _ = signatures(sorted(V2.glob('test-*.plans.jsonl')))
ban_g, ban_s = set(dg) | set(tg), set(ds) | set(ts)
def sig(c): return hashlib.sha256(js.dumps(c, ensure_ascii=False, separators=(',', ':')).encode()).hexdigest()
keep = [r for r in out if not (r.get('group') in ban_g or sig(r['chunks']) in ban_s)]
dst = V2 / 'v2extra' / 'v2.gemma.plans.jsonl'
dst.parent.mkdir(exist_ok=True)
dst.write_text(''.join(js.dumps(r, ensure_ascii=False) + chr(10) for r in keep), encoding='utf-8')
print(f'v2.gemma: keep {len(keep)}/{len(out)}')


In [ ]:
import os
ex = '/content/traind/data/prepared/v2extra/syn2.para.plans.jsonl,/content/traind/data/prepared/v2extra/syn3.para.plans.jsonl,/content/traind/data/prepared/v2extra/syn4.para.plans.jsonl,/content/traind/data/prepared/v2extra/train.para.plans.jsonl,/content/traind/data/prepared/v2extra/syn5.gemma.plans.jsonl,/content/traind/data/prepared/v2extra/syn6.qwen.plans.jsonl,/content/traind/data/prepared/v2extra/syn6.gemma.plans.jsonl,/content/traind/data/prepared/v2extra/v2.gemma.plans.jsonl'
os.environ.update({
    'INIT': 'hf', 'JEVA': '0.5', 'ALPHA': '0.15', 'MAXTEXTS': '80',
    'EXTRA': ex, 'PYTHONUTF8': '1', 'TORCH_DEVICE': dev,
})
%cd /content/traind/scripts/distill
!python -u /content/traind/scripts/distill/train_embed.py /content/traind/data/hf/turbo /content/traind/models /content/traind/data/prepared/v2 /content/traind/models/s1-distilled-d3.dte 3


In [ ]:
from google.colab import files
files.download('/content/traind/models/s1-distilled-d3.dte')
